# exp_038 — Targeted Re-inference Cascade

Re-runs only the 95 private questions that still have no `\boxed{}` after exp_037.
Three stages, first-with-boxed wins; merges results into a full 943-row submission.

## How to run
1. Attach the `151b-experiments` Kaggle dataset (must include
   `experiments/exp_037_multibox_v2/private_responses.jsonl` AND
   `experiments/exp_038_missing_boxed_cascade/targets.jsonl`)
2. GPU: **T4 x2**
3. Save Version → Save & Run All (Commit)

Runtime: ~10 min model load + ~30 min stage 1 + ~30 min stage 2 + ~5 min stage 3 = **~75 min**.

Output files in `/kaggle/working/`:
- `private_responses.jsonl` — full 943-row merged set (exp_037 + cascade winners)
- `submission.csv` — ready to submit to Kaggle
- `cascade_stats.json` — per-stage recovery counts

## 1. Install vLLM + libcuda symlink
After running, **restart the session** once.

In [ ]:
!pip install -q vllm transformers tqdm "antlr4-python3-runtime==4.11.1" "protobuf<6.0"

import subprocess, os

KNOWN_PATHS = [
    "/usr/local/nvidia/lib64/libcuda.so.1",
    "/usr/lib/x86_64-linux-gnu/libcuda.so.1",
    "/usr/lib/libcuda.so.1",
    "/lib/x86_64-linux-gnu/libcuda.so.1",
]
out = subprocess.run(["ldconfig", "-p"], capture_output=True, text=True).stdout
ldconfig_cands = [l.split("=>")[1].strip() for l in out.splitlines() if "libcuda.so.1" in l]
all_cands = KNOWN_PATHS + ldconfig_cands
libcuda = next((p for p in all_cands if os.path.exists(p) and "stubs" not in p), None) \
       or next((p for p in all_cands if os.path.exists(p)), None)
if not libcuda:
    result = subprocess.run(["find", "/usr", "/lib", "-name", "libcuda.so*", "-not", "-path", "*/stubs/*"],
                            capture_output=True, text=True)
    found = [p.strip() for p in result.stdout.splitlines() if p.strip()]
    libcuda = found[0] if found else None
if not libcuda:
    nvsmi = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    raise AssertionError(
        f"libcuda.so.1 not found. nvidia-smi rc={nvsmi.returncode}. "
        f"Check Settings → Accelerator: T4 x2.\n"
        f"nvidia-smi stdout: {nvsmi.stdout[:300]!r}"
    )
print(f"Real libcuda: {libcuda}")
stub_dir = "/kaggle/working/cuda_stubs"
os.makedirs(stub_dir, exist_ok=True)
stub = f"{stub_dir}/libcuda.so"
if os.path.lexists(stub): os.remove(stub)
os.symlink(libcuda, stub)
for var in ("LIBRARY_PATH", "LD_LIBRARY_PATH"):
    os.environ[var] = f"{stub_dir}:{os.environ.get(var, '')}"
subprocess.run(["rm", "-rf", "/root/.cache/flashinfer"], check=False)
print("Install complete. → Run → Restart session, then run the next cell.")

## 2. Imports + config + load targets
Uses base `Qwen/Qwen3-4B-Thinking-2507` (NOT the GRPO variant — GRPO chains are 2-4× longer).

In [ ]:
import json, os, re, glob
from pathlib import Path

os.environ["VLLM_USE_V1"] = "0"
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
_stub_dir = "/kaggle/working/cuda_stubs"
if os.path.isdir(_stub_dir):
    for _v in ("LIBRARY_PATH", "LD_LIBRARY_PATH"):
        os.environ[_v] = f"{_stub_dir}:{os.environ.get(_v, '')}"

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

def _find(filename):
    for c in [filename, *glob.glob(f"/kaggle/input/**/{filename}", recursive=True)]:
        if os.path.exists(c):
            return c
    return None

TARGETS_FILE = _find("experiments/exp_038_missing_boxed_cascade/targets.jsonl")
EXP037_FILE  = _find("experiments/exp_037_multibox_v2/private_responses.jsonl")
PRIVATE_FILE = _find("private.jsonl") or "/kaggle/input/competitions/cse-151-b-spring-2026-competition/private.jsonl"
assert TARGETS_FILE, "targets.jsonl not found — refresh the 151b-experiments Kaggle dataset"
assert EXP037_FILE,  "exp_037 private_responses.jsonl not found in dataset"
print(f"Targets:     {TARGETS_FILE}")
print(f"exp_037:     {EXP037_FILE}")
print(f"private set: {PRIVATE_FILE}")

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
VLLM_TP = 2; VLLM_MAX_MODEL_LEN = 10240; VLLM_MAX_NUM_SEQS = 32; VLLM_GPU_MEM_UTIL = 0.90
WORKING = Path("/kaggle/working")
OUT_RESPONSES = WORKING / "private_responses.jsonl"
OUT_SUBMISSION = WORKING / "submission.csv"
OUT_STATS = WORKING / "cascade_stats.json"

targets = [json.loads(l) for l in open(TARGETS_FILE)]
n_mcq = sum(1 for t in targets if t['is_mcq'])
n_ff = sum(1 for t in targets if not t['is_mcq'])
print(f"\nLoaded {len(targets)} target questions  (FF={n_ff}, MCQ={n_mcq})")

## 3. Load model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    dtype="float16",
    tensor_parallel_size=VLLM_TP,
    disable_custom_all_reduce=True,
    enable_prefix_caching=False,
    gpu_memory_utilization=VLLM_GPU_MEM_UTIL,
    max_model_len=VLLM_MAX_MODEL_LEN,
    trust_remote_code=True,
    max_num_seqs=VLLM_MAX_NUM_SEQS,
)
print("Model loaded.")

BOX_RE = re.compile(r"\\boxed\{")
def has_box(text):
    return bool(BOX_RE.search(text or ""))

## 4. Stage 1 — greedy + suppressed-thinking prompt
Force `<think>\n</think>\n\n` in the assistant turn so the model skips the thinking explosion entirely.

In [ ]:
def build_stage1_prompt(target):
    q = target['question']
    if target['is_mcq']:
        opts = target.get('options') or []
        opts_text = "\n".join(opts) if isinstance(opts, list) else str(opts)
        user_msg = f"Solve this multiple-choice problem. Reply with a single letter in \\boxed{{}}.\n\n{q}\n\nOptions:\n{opts_text}"
    else:
        user_msg = f"Solve this math problem. Put your final answer in \\boxed{{}}. Be concise.\n\n{q}"
    messages = [{"role": "user", "content": user_msg}]
    base = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    # Suppress thinking by pre-filling an empty <think>...</think>
    return base + "<think>\n\n</think>\n\n"

stage1_prompts = [build_stage1_prompt(t) for t in targets]
stage1_params = SamplingParams(n=1, max_tokens=2048, temperature=0.0, top_p=1.0)
print(f"Stage 1: greedy + suppressed-thinking on {len(stage1_prompts)} prompts")
stage1_outs = llm.generate(stage1_prompts, sampling_params=stage1_params)

stage1_winners = {}
for t, out in zip(targets, stage1_outs):
    text = out.outputs[0].text
    full = "<think>\n\n</think>\n\n" + text
    if has_box(full):
        stage1_winners[t['id']] = full

print(f"Stage 1 winners: {len(stage1_winners)}/{len(targets)}")
stage2_targets = [t for t in targets if t['id'] not in stage1_winners]
print(f"Stage 2 targets (residual): {len(stage2_targets)}")

## 5. Stage 2 — multi-sample (T=0.6, n=4, first-with-`\boxed{}` wins)
Different mechanism: sampling diversity. Some chains terminate while others explode; we take any that finished.

In [ ]:
def build_stage2_prompt(target):
    q = target['question']
    if target['is_mcq']:
        opts = target.get('options') or []
        opts_text = "\n".join(opts) if isinstance(opts, list) else str(opts)
        user_msg = f"{q}\n\nOptions:\n{opts_text}\n\nPut the letter of your answer in \\boxed{{}}."
    else:
        user_msg = f"{q}\n\nPut your answer in \\boxed{{}}."
    messages = [{"role": "user", "content": user_msg}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

stage2_winners = {}
if stage2_targets:
    stage2_prompts = [build_stage2_prompt(t) for t in stage2_targets]
    stage2_params = SamplingParams(n=4, max_tokens=4096, temperature=0.6, top_p=0.95, top_k=20)
    print(f"Stage 2: n=4 samples @ T=0.6 on {len(stage2_prompts)} prompts")
    stage2_outs = llm.generate(stage2_prompts, sampling_params=stage2_params)
    for t, out in zip(stage2_targets, stage2_outs):
        for o in out.outputs:
            if has_box(o.text):
                stage2_winners[t['id']] = o.text
                break
print(f"Stage 2 winners: {len(stage2_winners)}/{len(stage2_targets)}")
stage3_targets = [t for t in stage2_targets if t['id'] not in stage2_winners and t['is_mcq']]
print(f"Stage 3 targets (residual MCQ): {len(stage3_targets)}")

## 6. Stage 3 — letter-only forced MCQ
Last-resort prompt forces an MCQ guess. Even a uniform-random guess on 10 options scores ~10%.

In [ ]:
def build_stage3_prompt(target):
    q = target['question']
    opts = target.get('options') or []
    opts_text = "\n".join(opts) if isinstance(opts, list) else str(opts)
    user_msg = (
        f"{q}\n\nOptions:\n{opts_text}\n\n"
        f"Respond with EXACTLY one letter (A-J) in \\boxed{{}}. No reasoning, no explanation."
    )
    messages = [{"role": "user", "content": user_msg}]
    base = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return base + "<think>\n\n</think>\n\n"

stage3_winners = {}
if stage3_targets:
    stage3_prompts = [build_stage3_prompt(t) for t in stage3_targets]
    stage3_params = SamplingParams(n=4, max_tokens=512, temperature=0.6, top_p=0.95, top_k=20)
    print(f"Stage 3: letter-only n=4 on {len(stage3_prompts)} prompts")
    stage3_outs = llm.generate(stage3_prompts, sampling_params=stage3_params)
    for t, out in zip(stage3_targets, stage3_outs):
        for o in out.outputs:
            full = "<think>\n\n</think>\n\n" + o.text
            if has_box(full):
                stage3_winners[t['id']] = full
                break
print(f"Stage 3 winners: {len(stage3_winners)}/{len(stage3_targets)}")

## 7. Merge with exp_037 + write submission

In [ ]:
exp037 = {r['id']: r for r in (json.loads(l) for l in open(EXP037_FILE))}
# Each stage only ran on the previous stage's residual, so these dicts have disjoint keys.
all_winners = {**stage1_winners, **stage2_winners, **stage3_winners}

merged = []
n_replaced = 0
for qid in sorted(exp037.keys()):
    r = dict(exp037[qid])
    if qid in all_winners:
        r['response'] = all_winners[qid]
        n_replaced += 1
    merged.append(r)

with open(OUT_RESPONSES, "w") as f:
    for r in merged:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"Wrote {OUT_RESPONSES}  ({len(merged)} rows; {n_replaced} replaced)")

import csv
with open(OUT_SUBMISSION, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["id", "response"])
    for r in merged:
        w.writerow([r['id'], r['response']])
print(f"Wrote {OUT_SUBMISSION}")

stats = {
    "targets": {"total": len(targets), "ff": n_ff, "mcq": n_mcq},
    "stage1_winners": len(stage1_winners),
    "stage2_winners": len(stage2_winners),
    "stage3_winners": len(stage3_winners),
    "total_replaced": n_replaced,
    "still_missing": len(targets) - n_replaced,
}
with open(OUT_STATS, "w") as f:
    json.dump(stats, f, indent=2)
print("\n=== cascade_stats ===")
print(json.dumps(stats, indent=2))

## 8. Preview a few cascade winners

In [ ]:
import random
random.seed(0)
winner_ids = list(all_winners.keys())
for qid in random.sample(winner_ids, min(3, len(winner_ids))):
    stage = "1" if qid in stage1_winners else ("2" if qid in stage2_winners else "3")
    t = next(t for t in targets if t['id'] == qid)
    print(f"\n── id={qid}  stage={stage}  is_mcq={t['is_mcq']} ──")
    print(f"Q: {t['question'][:200]}...")
    print(f"\nResponse (last 600 chars):")
    print(all_winners[qid][-600:])
    print("─" * 60)